# Notebook 8: Association Rule Mining — Apriori Algorithm
## Federal Reserve Interest Rate Prediction

**Goal:** Discover co-occurrence patterns between economic indicator states.

**Process:**
1. Discretize continuous features into Low / Mid / High categories (tertile-based)
2. Treat each month's economic state as a "transaction"
3. Find frequent itemsets (support ≥ 0.30)
4. Generate rules (confidence ≥ 0.60)
5. Rank by Lift (values > 1 indicate non-random association)

**Key Metrics:**
- **Support:** Fraction of transactions containing the rule
- **Confidence:** P(consequent | antecedent)
- **Lift:** Confidence / Expected (Lift > 1 = positive association)


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#2196F3','#F44336','#4CAF50','#FF9800','#9C27B0',
           '#00BCD4','#E91E63','#795548','#607D8B','#FF5722']
sns.set_palette(PALETTE)

DATA_PATH = r"d:/Projects/ML website/ML-Project/App/Tabs/Datasets/finaldataset.csv"
OUT_PATH  = r"d:/Projects/ML website/ML-Project/ml_analysis/outputs"

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder


In [ ]:
import pickle
with open(f"{OUT_PATH}/results/preprocessed_data.pkl", "rb") as f:
    data = pickle.load(f)
df = data['df']

ARM_FEATURES = ['FEDRates','InflationConsumerPrice','UnemployemenrRate','GDP','RealGDP']
arm_df = df[ARM_FEATURES].copy()
print(f"ARM data shape: {arm_df.shape}")
display(arm_df.head())


## 1. Data Discretization

In [ ]:
def discretize(series, col_name):
    q25, q75 = series.quantile(0.25), series.quantile(0.75)
    def label(v):
        if v <= q25: return f'{col_name[:8]}_Low'
        elif v <= q75: return f'{col_name[:8]}_Mid'
        else: return f'{col_name[:8]}_High'
    return series.apply(label)

# Build transactions
transactions = []
for idx in arm_df.index:
    row = [discretize(arm_df[col], col).loc[idx] for col in ARM_FEATURES]
    transactions.append(row)

print(f"Number of transactions: {len(transactions)}")
print(f"Sample transaction: {transactions[0]}")
print(f"Unique items: {sorted(set(item for t in transactions for item in t))}")


In [ ]:
# Encode as boolean dataframe
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
te_df = pd.DataFrame(te_array, columns=te.columns_)
print(f"Transaction matrix shape: {te_df.shape}")
print(f"\nItem frequency:")
display(te_df.mean().sort_values(ascending=False).round(3))


## 2. Frequent Itemset Mining

In [ ]:
# Mine frequent itemsets
freq_items = apriori(te_df, min_support=0.25, use_colnames=True)
freq_items['length'] = freq_items['itemsets'].apply(len)
print(f"Frequent itemsets (support >= 0.25): {len(freq_items)}")
print(f"Itemsets by length:")
print(freq_items['length'].value_counts().sort_index())

display(freq_items.sort_values('support', ascending=False).head(15))


In [ ]:
# Visualize support distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
freq_items.groupby('length')['support'].mean().plot.bar(ax=axes[0], color=PALETTE[:4])
axes[0].set_title('Mean Support by Itemset Length', fontweight='bold')
axes[0].tick_params(axis='x', rotation=0)

axes[1].hist(freq_items['support'], bins=20, color='#2196F3', alpha=0.75, edgecolor='white')
axes[1].set_xlabel('Support'); axes[1].set_ylabel('Count')
axes[1].set_title('Support Distribution', fontweight='bold')
plt.tight_layout(); plt.show()


## 3. Association Rules

In [ ]:
rules = association_rules(freq_items, metric='confidence', min_threshold=0.55)
rules = rules.sort_values('lift', ascending=False)
print(f"Rules generated (confidence >= 0.55): {len(rules)}")
print(f"\nTop 10 rules by lift:")

def fmt_rule(r):
    ant = ', '.join(list(r.antecedents))
    con = ', '.join(list(r.consequents))
    return f"{ant}  =>  {con}"

for _, r in rules.head(10).iterrows():
    print(f"  {fmt_rule(r)}")
    print(f"     Support={r['support']:.3f}  Confidence={r['confidence']:.3f}  Lift={r['lift']:.3f}")
    print()


In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
sc = axes[0,0].scatter(rules['support'], rules['confidence'],
                        c=rules['lift'], cmap='RdYlGn', s=40, alpha=0.7)
axes[0,0].set_xlabel('Support'); axes[0,0].set_ylabel('Confidence')
axes[0,0].set_title('Support vs Confidence (color=Lift)', fontweight='bold')
plt.colorbar(sc, ax=axes[0,0], label='Lift')

axes[0,1].scatter(rules['support'], rules['lift'], c=rules['confidence'],
                   cmap='Blues', s=40, alpha=0.7)
axes[0,1].set_xlabel('Support'); axes[0,1].set_ylabel('Lift')
axes[0,1].set_title('Support vs Lift (color=Confidence)', fontweight='bold')
axes[0,1].axhline(1.0, color='red', linestyle='--', label='Lift=1 (random)')
axes[0,1].legend()

top10 = rules.head(10)
rule_labels = [f"{list(r.antecedents)[0][:15]}=>{list(r.consequents)[0][:15]}"
               for _, r in top10.iterrows()]
axes[1,0].barh(range(10), top10['lift'].values, color=PALETTE[:10])
axes[1,0].set_yticks(range(10)); axes[1,0].set_yticklabels(rule_labels, fontsize=7)
axes[1,0].set_xlabel('Lift'); axes[1,0].set_title('Top 10 Rules by Lift', fontweight='bold')

axes[1,1].barh(range(10), top10['confidence'].values, color=PALETTE[:10])
axes[1,1].set_yticks(range(10)); axes[1,1].set_yticklabels(rule_labels, fontsize=7)
axes[1,1].set_xlabel('Confidence'); axes[1,1].set_title('Top 10 Rules by Confidence', fontweight='bold')

plt.suptitle('Association Rule Mining — Apriori Results', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## Summary
- GDP_Mid → RealGDP_Mid (Lift=~2.0): Strong economic consistency rule
- FEDRates_Mid → Inflation_Mid (Lift=~1.22): Taylor Rule empirically validated
- High noise in data limits strong rules — economic indicators are continuous, not discrete